# Read the description of the current state

Read the label and a domain field directly from the selected status object.

## Prepare the model

Run this cell first. It imports the required types and defines the model used by this example. Start with a fresh repository Python kernel so declarations from other examples do not enter this model.

In [1]:
from pydantic import Field

from aoa.action_machine.domain import BaseStateEntity, Lifecycle, value


class OrderStateEntity(BaseStateEntity):
    """Describe one possible order status."""

    requires_receipt: bool = Field(default=False, description="Whether a receipt is needed")

    DRAFT = value(code="draft", display_name="Draft")
    PAID = value(code="paid", display_name="Payment received", requires_receipt=True)



class OrderLifecycle(Lifecycle[OrderStateEntity]):
    """Collect statuses and declare their allowed transition."""

    _template = Lifecycle().state(OrderStateEntity.DRAFT).to(OrderStateEntity.PAID).initial().state(OrderStateEntity.PAID).final()

## Run the example

The following cell performs the operation described above. Its assertions check the result; any expected refusal is caught explicitly.

In [2]:
row = {"id": "ord-1", "status": "paid"}
current = OrderLifecycle(row["status"])
status = current.current_status
assert status is OrderStateEntity.PAID
assert current.current_state == status.code
assert status.requires_receipt is True
assert OrderStateEntity.entity_type is OrderStateEntity
assert tuple(OrderStateEntity.by_code(code) for code in OrderStateEntity.codes()) == OrderStateEntity.members
print("codes:", OrderStateEntity.codes())
print("current code:", current.current_state)
print("label:", status.display_name)
print("requires receipt:", status.requires_receipt)
print("declared object:", status is OrderStateEntity.PAID)

codes: ['draft', 'paid']
current code: paid
label: Payment received
requires receipt: True
declared object: True


## What the result means

The constructor takes the code from the row and stores a reference to the declared status object. current_status returns that exact object; current_state reads its code without storing a second current code. The requires_receipt field is domain data, not an instruction to issue a receipt. StateVocabulary.by_code remains useful when looking up an arbitrary code rather than the current state.